# Stage C — Label-holdout discovery evaluation

Evaluate mask-holdout Stage B checkpoints. Holdout stream *labels* were never in the loss.

| Model | Meaning |
|-------|---------|
| **mock** | Mock-pretrained weights, no real FT |
| **lastblock_ft** | Freeze backbone; train last CPEN block + decoders |
| **full_ft** | Full fine-tune (optional baseline) |

| Tier | What it measures |
|------|------------------|
| **1. Controlled recovery** | Recall / mean $p$ on holdout members, **solo** vs **co-located** |
| **2. L1SO summary** | Optional leave-one-stream-out folds |
| **3. Open discovery** | High-$p$ Background islands — candidates, not FPs |

Default holdouts: AAU, Elqui, Chenab, Indus, Phoenix.

Prefer the batch compare script (writes under `figures/real_label_holdout_eval/`):
```bash
PYTHONPATH=src:. python scripts/stage_c_compare_holdout.py \
  --mock-run-dir …/blobs/…/edgecons \
  --lastblock-run-dir …/blobs_real_mask_lastblock/…/lastblock1-… \
  --ft-run-dir …/blobs_real_mask/…/maskholdout-…
```
Or run the cells below for a single `RUN_DIR`.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.lines import Line2D

REPO = Path.cwd().resolve()
for cand in (REPO, *REPO.parents):
    if (cand / "src" / "cpen").is_dir():
        REPO = cand
        break
sys.path[:0] = [str(REPO), str(REPO / "src")]

from cpen.apps.streams.cell_stream_split import describe_cell_split
from cpen.apps.streams.preprocess.config import DEFAULT_REAL_PIPELINE_ROOT, TEST_REAL_STREAMS
from cpen.apps.streams.real_discovery import (
    DiscoveryConfig,
    apply_threshold,
    choose_threshold_supervised,
    fit_stats_for_cfg,
    galaxy_dir,
    holdout_recovery_breakdown,
    load_lit_from_run,
    open_discovery_candidates,
    rebuild_cell_split,
    score_manifest,
    stream_recovery_table,
)

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False, "font.size": 11})
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("repo", REPO, "device", DEVICE)


## 0. Point at a Stage-B run (or load compare table)

Auto-selects the newest `best.ckpt` under `blobs_real_mask`. For last-block FT, set
`SWEEP_ROOT` to `blobs_real_mask_lastblock` or override `RUN_DIR`.

If `holdout_recovery_compare.parquet` already exists from the Slurm Stage-C step, the
next cell can display that head-to-head without re-scoring.


In [ ]:
from datetime import datetime

SWEEP_ROOT = Path(
    "/scratch/gpfs/BHANIN/jdezoort/cpen_runs/blobs_real_mask/stream_output/adamw/sweep_lr"
)
LASTBLOCK_ROOT = Path(
    "/scratch/gpfs/BHANIN/jdezoort/cpen_runs/blobs_real_mask_lastblock/stream_output/adamw/sweep_lr"
)
MOCK_RUN_DIR = Path(
    "/scratch/gpfs/BHANIN/jdezoort/cpen_runs/blobs/stream_output/adamw/sweep_lr/"
    "cpen_4_256_0p2_t5_ln_cosine_op-incidence_onorm-degree_res-transformer-like_drop0p2_nf6_"
    "gnativeedges_blobs-dropparallax-hyper-edgecons"
)
COMPARE_PARQUET = (
    REPO / "notebooks" / "streams" / "figures" / "real_label_holdout_eval"
    / "holdout_recovery_compare.parquet"
)

def _newest_run(root: Path, substr: str = "maskholdout") -> Path | None:
    cands = []
    if not root.is_dir():
        return None
    for d in root.iterdir():
        if not d.is_dir() or substr not in d.name:
            continue
        ck = d / "best.ckpt"
        if ck.is_file():
            cands.append((ck.stat().st_mtime, d))
    if not cands:
        return None
    cands.sort(reverse=True)
    return cands[0][1]

# Prefer last-block FT if present; else full mask FT
RUN_DIR = _newest_run(LASTBLOCK_ROOT, "lastblock") or _newest_run(SWEEP_ROOT, "maskholdout")
if RUN_DIR is None:
    raise FileNotFoundError("No maskholdout/lastblock run with best.ckpt found")

print("RUN_DIR", RUN_DIR)
print("mtime", datetime.fromtimestamp((RUN_DIR / "best.ckpt").stat().st_mtime))
if COMPARE_PARQUET.is_file():
    cmp = pd.read_parquet(COMPARE_PARQUET)
    print("\n=== existing compare table ===")
    display(cmp[cmp.subset == "all"][["model", "stream_label", "n_members", "recall", "mean_p"]])


## 1. Rebuild split + load model + score galaxy

Loads `best.ckpt` (monitor = `val_sup_auroc` on current Stage-B jobs). Safe to score while the job is still running.


In [ ]:
CELL_SPLIT = rebuild_cell_split(CFG)
print(describe_cell_split(CELL_SPLIT))
print(
    "Note: under mask-holdout there is no discovery *cell* split.\n"
    "  fit_val   = cells held out of the optimizer (early stopping)\n"
    "  fit_train = every other cell\n"
    "Holdout recovery is by stream_label (AAU/…), not by role==test."
)
STATS, _, train_man = fit_stats_for_cfg(CFG)
print(f"feature stats from {len(train_man):,} train-pool graphs")

LIT, CKPT, HP, DEVICE = load_lit_from_run(RUN_DIR, cfg=CFG, device=DEVICE)
print("ckpt", CKPT, "hparams", HP)

MAX_GRAPHS = None  # set e.g. 50 for a dry run
SCORED = score_manifest(
    CFG, lit=LIT, stats=STATS, cell_split=CELL_SPLIT,
    device=DEVICE, max_graphs=MAX_GRAPHS, progress=True,
)
print(f"scored {len(SCORED):,} blobs")
print("fit-split roles (not a discovery split):")
print(pd.Series([r["role"] for r in SCORED]).value_counts())


## 2. Threshold on supervised (non-holdout) fit_val stars only

Uses stars in **fit_val** cells whose labels are not in the holdout set.


In [ ]:
T_STAR, SWEEP = choose_threshold_supervised(
    SCORED, holdout_streams=HOLDOUT, roles=["fit_val", "val"]
)
print(f"t* (max F1 on supervised val) = {T_STAR:.3f}")
display(SWEEP.sort_values("f1", ascending=False).head(5))

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(SWEEP["t"], SWEEP["f1"], label="F1", lw=2)
ax.plot(SWEEP["t"], SWEEP["prec"], label="precision", alpha=0.8)
ax.plot(SWEEP["t"], SWEEP["rec"], label="recall", alpha=0.8)
ax.axvline(T_STAR, color="k", ls="--", lw=1, label=f"t*={T_STAR:.2f}")
ax.set_xlabel("threshold"); ax.set_ylabel("score")
ax.set_title("Threshold on supervised val stars (holdouts excluded)")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

STARS = apply_threshold(SCORED, T_STAR)
print(f"stars={len(STARS):,}  pred+={(STARS.pred).sum():,}  S5+={STARS.y.sum():,}")


## 3. Tier 1 — Controlled holdout recovery (solo vs co-located)

**Solo** = holdout is the only catalog stream in that cell (clean discovery).  
**Co-located** = shares the cell with train streams (crowded-field stress test).


In [ ]:
OCC = CELL_SPLIT.occupancy
rec = holdout_recovery_breakdown(STARS, holdout_streams=HOLDOUT, occupancy=OCC)
display(rec)

# Bar chart: recall by stream × subset
fig, ax = plt.subplots(figsize=(9, 4.5))
streams = sorted(HOLDOUT)
x = np.arange(len(streams))
w = 0.35
for i, subset in enumerate(("solo", "colocated")):
    vals = []
    for s in streams:
        row = rec[(rec.stream_label == s) & (rec.subset == subset)]
        vals.append(float(row.recall.iloc[0]) if len(row) and pd.notna(row.recall.iloc[0]) else 0.0)
    ax.bar(x + (i - 0.5) * w, vals, width=w, label=subset)
ax.set_xticks(x); ax.set_xticklabels(streams, rotation=20, ha="right")
ax.set_ylim(0, 1.05); ax.set_ylabel("recall @ t*")
ax.set_title("Holdout recovery: solo vs co-located")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

# Sky: holdout TP vs FN
fig, ax = plt.subplots(figsize=(8, 5))
hold = STARS[STARS.stream_label.isin(HOLDOUT)]
ax.scatter(hold.loc[hold.tp, "ra"], hold.loc[hold.tp, "dec"], s=10, c="lime", label="TP", alpha=0.85)
ax.scatter(hold.loc[hold.fn, "ra"], hold.loc[hold.fn, "dec"], s=10, c="crimson", marker="x", label="FN", alpha=0.85)
ax.invert_xaxis(); ax.set_xlabel("RA"); ax.set_ylabel("Dec")
ax.set_title(f"Holdout members @ t*={T_STAR:.2f}")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()


## 4. Per-holdout zoom (3-panel)

For each holdout stream: catalog members | model score | triage.


In [ ]:
def plot_stream_zoom(name: str, stars: pd.DataFrame = STARS, pad: float = 0.8):
    sub = stars[stars.stream_label == name]
    if not len(sub):
        print(name, "absent"); return
    # include nearby field for context
    ra0, dec0 = sub.ra.mean(), sub.dec.mean()
    near = stars[(stars.ra - ra0).abs().lt(pad) & (stars.dec - dec0).abs().lt(pad)]
    fig, axs = plt.subplots(1, 3, figsize=(13, 4), sharex=True, sharey=True)
    for ax in axs:
        ax.scatter(near.loc[~near.y, "ra"], near.loc[~near.y, "dec"], s=1, c="#e8e8e8", rasterized=True)
        ax.invert_xaxis(); ax.set_xlabel("RA")
    axs[0].scatter(sub.ra, sub.dec, s=8, c="#d95f02")
    axs[0].set_title(f"{name} catalog"); axs[0].set_ylabel("Dec")
    sc = axs[1].scatter(near.ra, near.dec, s=4, c=near.p, cmap="magma", vmin=0, vmax=1, rasterized=True)
    fig.colorbar(sc, ax=axs[1], fraction=0.046, pad=0.02, label="p")
    axs[1].set_title("model score")
    axs[2].scatter(sub.loc[sub.tp, "ra"], sub.loc[sub.tp, "dec"], s=10, c="lime", label="TP")
    axs[2].scatter(sub.loc[sub.fn, "ra"], sub.loc[sub.fn, "dec"], s=10, c="crimson", label="FN")
    axs[2].set_title("holdout triage"); axs[2].legend(frameon=False, fontsize=8)
    fig.suptitle(f"{name}  N={len(sub)}  recall={sub.tp.mean():.2f}  mean p={sub.p.mean():.3f}", y=1.02)
    plt.tight_layout(); plt.show()

for name in sorted(HOLDOUT):
    plot_stream_zoom(name)


## 5. Tier 2 — L1SO run summary (optional)

If you launched `train_blob_real_ft_l1so.slurm`, set `L1SO_ROOT` and this cell aggregates per-fold `best` metrics from parquet / run names.


In [ ]:
L1SO_ROOT = Path("/scratch/gpfs/BHANIN/jdezoort/cpen_runs/blobs_real_l1so/stream_output/adamw/sweep_lr")
rows = []
if L1SO_ROOT.is_dir():
    for p in sorted(L1SO_ROOT.glob("*maskholdout*l1so*")):
        pq = Path(str(p) + ".parquet")
        if not pq.is_file():
            # sibling parquet sometimes next to dir
            alt = p.parent / (p.name + ".parquet")
            pq = alt if alt.is_file() else pq
        fold = None
        for tok in p.name.split("_"):
            if tok.startswith("l1so"):
                fold = tok[len("l1so"):]
        meta = {"run": p.name, "fold_tag": fold, "has_ckpt": (p / "best.ckpt").is_file()}
        if pq.is_file():
            try:
                tab = pd.read_parquet(pq)
                if "val_auroc" in tab.columns:
                    meta["best_val_auroc"] = float(tab["val_auroc"].max())
            except Exception as exc:
                meta["err"] = str(exc)
        rows.append(meta)
    display(pd.DataFrame(rows))
else:
    print("L1SO_ROOT not found — skip (submit train_blob_real_ft_l1so.slurm first)")


## 6. Tier 3 — Open discovery candidates

Predicted-positive **Background** stars grouped by cell. These are *candidates*, not false positives.


In [ ]:
cands = open_discovery_candidates(STARS, threshold=T_STAR, min_stars=30, min_mean_p=0.25)
print(f"{len(cands)} candidate cells")
display(cands.head(15))

def plot_candidate_cell(cell_id: int):
    sub = STARS[STARS.cell_id == int(cell_id)]
    fig, axs = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
    axs[0].scatter(sub.loc[~sub.y, "ra"], sub.loc[~sub.y, "dec"], s=2, c="#ddd", rasterized=True)
    s5 = sub[sub.y]
    if len(s5):
        for name, g in s5.groupby("stream_label"):
            axs[0].scatter(g.ra, g.dec, s=8, label=name)
        axs[0].legend(frameon=False, fontsize=7)
    axs[0].set_title(f"cell {cell_id} catalog"); axs[0].invert_xaxis(); axs[0].set_ylabel("Dec")
    sc = axs[1].scatter(sub.ra, sub.dec, s=5, c=sub.p, cmap="magma", vmin=0, vmax=1, rasterized=True)
    fig.colorbar(sc, ax=axs[1], fraction=0.046, pad=0.02)
    axs[1].set_title("model score"); axs[1].invert_xaxis()
    for ax in axs:
        ax.set_xlabel("RA")
    plt.tight_layout(); plt.show()

for cid in cands.cell_id.head(6):
    plot_candidate_cell(int(cid))


## 7. Export

In [ ]:
OUT = REPO / "notebooks" / "streams" / "figures" / "real_label_holdout_eval"
OUT.mkdir(parents=True, exist_ok=True)
rec.to_parquet(OUT / "holdout_recovery.parquet", index=False)
cands.to_parquet(OUT / "open_candidates.parquet", index=False)
STARS[["cell_id", "role", "ra", "dec", "p", "y", "stream_label", "pred", "tp", "fp", "fn"]].to_parquet(
    OUT / f"stars_t{T_STAR:.2f}.parquet", index=False
)
meta = {
    "run_dir": str(RUN_DIR),
    "ckpt": str(CKPT),
    "t_star": T_STAR,
    "holdout_streams": sorted(HOLDOUT),
    "protocol": "mask_holdout",
}
import json as _json
(OUT / "last_eval_meta.json").write_text(_json.dumps(meta, indent=2) + "\n")
print("wrote", OUT)
